source: https://huggingface.co/spaces/manu02/DINOv3-Interactive-Patch-Cosine-Similarity/tree/main

In [ ]:
# If needed, install interactive backend & widgets (uncomment as needed):
# %pip install -q ipympl ipywidgets

# Enable interactive Matplotlib for clicks & live updates:
%matplotlib ipympl
# %matplotlib widget

In [ ]:
from pathlib import Path
from os import listdir
from os.path import isfile, join
from dataclasses import dataclass
import random, math

import numpy as np
from PIL import Image
import torch
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import Rectangle
from matplotlib.image import AxesImage
from transformers import AutoModel

try:
    import ipywidgets as Widget
except Exception:
    Widget = None  # slider is optional
    

In [ ]:

# ---------- Row/col <-> idx utilities ----------
def rc_to_idx(r, c, cols): return int(r) * cols + int(c)
def idx_to_rc(i, cols):    return (int(i) // cols, int(i) % cols)

### Dataclasses

In [ ]:

# ---- raw image data ----

@dataclass
class RawImage:
    """
    Dataclass for raw/source images' metadata
    """
    image_id: str  # unique image_id
    path: str  # source of the raw image
    name: str  # name for reference

    def __init__(self, img_path: str):
        path = Path(img_path)
        if path.is_file():
            self.image_id = "/".join([path.parent.parent.name, path.parent.name, path.stem])
            self.path = path
            self.name = path.parent.name  #path.stem
        else:
            raise FileNotFoundError(f"Error: Path '{path}' does not exist.")
        
# ---- preprocessed ----

@dataclass
class NormalizationParameters:
    """Normalization parameters"""
    mean: int
    std: int 

@dataclass
class ImageTensor:
    """
    Dataclass for images preprocessed and ready as input tensor for pretrained encoder backbones (e.g. a pretrained DINOv3 ViT model) 
    """
    image_id: str  # unique image_id
    tensor: torch.Tensor
    width: int
    height: int
    channels: int
    patchsize: int
    norm: NormalizationParameters
    source: RawImage


@dataclass
class PreprocessedData:
    
    img_tensors: list[ImageTensor]
    n_images: int
    raw_imgs_metadata: list[RawImage]  # metadat of all raw/source images


# ---- encoding ----

@dataclass
class PatchEmbedding:
    """Embedding of a single patch of an image"""
    image_id: int
    patch_idx: int

    x: np.ndarray
    xn: np.ndarray


@dataclass
class ImageEmbedding:
    """Embedding of a single patch of an image"""
    image_id: int
    X: np.ndarray
    Xn: np.ndarray
    rows: int
    cols: int
    patchsize: int
    source: RawImage

    def get_patch_embedding(self, patch_idx) -> PatchEmbedding:
        
        # retrieve clicked embedding
        idx = self.clamp_idx(patch_idx)  # queried embedding idx
        x = self.X[idx]  # queried embedding
        # qn = src["Xn"][q_idx]  # Same? which is faster?
        xn = x / (np.linalg.norm(x) + 1e-8)  # normalized embedding (length = 1.0)  | linalg.norm = length

        return PatchEmbedding(self.image_id, idx, x, xn)
        # cos_self = np.matmul(src["Xn"], qn)  # same as: cos_self = src["Xn"] @ qn
        
        # n_patches = rows * cols  # 4096
        # patch_embs = hs[-n_patches: :].reshape(n_rows, n_cols, -1) # (64, 64, 384)

    def clamp_idx(self, i: int) -> int: 
        # makes sure i is one of the image patches (in range[0, r*c-1])
        return int(np.clip(i, 0, self.rows * self.cols - 1))
    
    def row(self, y: int) -> int:
        row = int(np.clip(y // self.patchsize, 0, self.rows - 1))
        return row

    def col (self, x: int) -> int:
        col = int(np.clip(x // self.patchsize, 0, self.cols - 1))
        return col
    
    def get_rows_and_cols(self):
        return self.rows, self.cols
    
    def get_n_patches(self) -> int:
        return self.cols * self.rows
    

@dataclass
class EmbeddingData:
    
    img_embs: list[ImageEmbedding]    # all image embeddings
    n_images: int
    raw_imgs_metadata: list[RawImage]  # metadat of all raw/source images
    pretrained_model_name: str


# ---- cosine similarity ----

@dataclass
class CosSimPatchImage:
    """Cosine similarity between single patch embedding and an image"""
    patch_emb: PatchEmbedding
    image_emb: ImageEmbedding
    cos_sim: np.ndarray
    top_n_ids: list[int]
    top_n_sims: list[float]

    def __init__(self, patch_emb: PatchEmbedding, image_emb: ImageEmbedding, top_n = 6):
        
       self.patch_emb = patch_emb
       self.image_emb = image_emb
       self.cos_sim = np.matmul(image_emb.Xn, patch_emb.xn)
       top_n_ids = [ int(i) for i in np.argpartition(self.cos_sim, -top_n)[-top_n:]]
       top_n_sims = [float(self.cos_sim[top_n_id]) for top_n_id in top_n_ids]
       # sort top_n_sims and top_n_ids by top_n_sims
       self.top_n_sims, self.top_n_ids = (list(reversed(t)) for t in zip(*sorted(zip(top_n_sims, top_n_ids))))

    def most_similar(self):
        best = int(np.argmax(self.cos_sim))
        return best
    

@dataclass
class CosineSimilarityData:
    """Cosine similarities between a patch-embedding and all the embeddings of all the images"""

    emb_data: EmbeddingData

    # selected image and patch
    img_selected_idx: int   # index of active/selected image
    patch_selected_idx: int  # index of selected patch (in img_selected_idx image) 

    # the (patch, images) cosine similarities
    patch_img: list[CosSimPatchImage]     # cosine similarities between patch_embedding and all image embeddings

    def __init__(self, emb_data: EmbeddingData):
        self.emb_data = emb_data
        self.img_selected_idx = None
        self.patch_selected_idx = None

        # ---- selected image and patch ----
        img_idx = 0  # first image
        img_emb = self.emb_data.img_embs[img_idx]
        row = img_emb.rows // 2  # select middle row
        col = img_emb.cols // 2  # select middle column
        patch_idx = rc_to_idx(row, col, img_emb.cols)
        
        # update cos-sims for selected patch
        self.update(img_idx, patch_idx)
        
    def update(self, img_idx: int, patch_idx: int):
        
        if img_idx != self.img_selected_idx or patch_idx != self.patch_selected_idx:

            # find new image and patch selected
            self.img_selected_idx = img_idx
            self.patch_selected_idx = patch_idx

            # update cosine_similarities for new patch
            print(f"Calculating cosine similarity between img[{self.img_selected_idx}]-patch[{self.patch_selected_idx}] and {len(self.emb_data.img_embs)} images ...", end="")
            img_emb_selected = self.emb_data.img_embs[self.img_selected_idx]
            patch_emb = img_emb_selected.get_patch_embedding(self.patch_selected_idx)
            self.patch_img = [ CosSimPatchImage(patch_emb, img_emb) for img_emb in self.emb_data.img_embs ]
            print(" Done")

    def top_n_img_idxs(self, n: int):
    
        # get top cosine similarity for each available image
        top_cos_sim_per_img = [cs.cos_sim.max().item() for cs in self.patch_img]
        # print(f"top_cos_sim_per_img: {top_cos_sim_per_img}")

        # get n image indices ordered by best (higest) cosine similarity
        top_n_cos_sim_img_idxs = [x[0] for x in reversed(sorted(enumerate(top_cos_sim_per_img), key=lambda x: x[1])[-n:])]

        return top_n_cos_sim_img_idxs


# ---- UI dataclasses ----

@dataclass
class ClickedIndices:
    ax_idx: int
    img_idx: int
    row_idx: int
    col_idx: int
    patch_idx: int

    def update_patch_idx(self, n_cols: int):
        self.patch_idx = rc_to_idx(self.row_idx, self.col_idx, n_cols)

    def is_undefined(self):
        return self.patch_idx is None or self.img_idx is None or self.ax_idx is None

### Data loading

### Data preprocessing

In [ ]:

# ---------- Image I/O ----------
def load_image(raw_img: RawImage) -> Image:
    """Load an image from PATH and return a PIL RGB image."""
    return Image.open(raw_img.path).convert("RGB")


# ---------- Preprocessing (custom, NO resize) ----------
def crop_to_patchsize(pil_img: Image, patchsize) -> Image:
    """Pad PIL image on right/bottom so (h,w) are multiples of `patchsize`."""
    w, h = pil_img.size
    h_crop = (h // patchsize) * patchsize
    w_crop = (w // patchsize) * patchsize
    return pil_img.crop((0, 0, w_crop, h_crop))
    
def preprocess_image(
        raw_img: RawImage, 
        device: torch.device = torch.device("cuda"),   # alt: use device("cpu")
        patchsize: int = 16, 
        mean=[0.485, 0.456, 0.406], # ImageNet dataset mean
        std=[0.229, 0.224, 0.225] # ImageNet dataset standard deviation
        ) -> ImageTensor:
    """Crop (right/bottom) -> ToTensor -> Normalize (default: ImageNet stats)."""

    pil_img = load_image(raw_img)
    img_cropped = crop_to_patchsize(pil_img, patchsize)
    transform = transforms.Compose([
        transforms.ToTensor(),  # CxHxW in range [0.0, 1.0]
        transforms.Normalize(mean, std)
    ])
    img_tensor = transform(img_cropped).unsqueeze(0).to(device)  # (1,3,height,width)
    batchsize, channels, height, width = img_tensor.shape

    return ImageTensor(
        raw_img.image_id, 
        img_tensor, 
        width, 
        height, 
        channels,
        patchsize,
        norm=NormalizationParameters(mean, std),
        source=raw_img,
    )

def preprocess(raw_img_paths: list[str], device, patchsize) -> PreprocessedData:
    print(f"Preprocessing {len(raw_img_paths)} raw images  ...", end="")
    raw_imgs_metadata = [ RawImage(path) for path in raw_img_paths]
    img_tensors = [ preprocess_image(raw_img, device, patchsize) for raw_img in raw_imgs_metadata ]
    print(" Done")
    return PreprocessedData(img_tensors, len(img_tensors), raw_imgs_metadata)

# img_tensors = preprocess(raw_img_paths, device, patchsize)


### Data Encoding

In [ ]:

# ---- model ----

def load_encoder_model(pretrained_model_name: str, device: torch.device):
    print(f"Loading pretrained '{pretrained_model_name}' encoding model")
    model = AutoModel.from_pretrained(pretrained_model_name).to(device)
    return model

# ---- encoding ----

def normalize_embedding(x: np.ndarray, order_norm: int =2, offset: float = 1e-8) -> np.ndarray:
    """
    Normalize each embedding (row) of the matrix X(,c) (to have unit length) by dividing by norm (default L2).

    Arguments
    ---------
    x
        A numpy matrix of shape (n, m)
    order_norm
        The order of the norm to be used, default 2 (L2-norm)
    offset
        Small offset to prevent division by zero
    
    Return
    ------
    x_normalized
        The normalized (by row) numpy matrix(rows, columns) of X.
    """
    assert len(x.shape) in (1,2), f"Unexpected matrix dimensionality of {len(x.shape)}, should be 1 or 2"
    x_norm = np.linalg.norm(x, ord=order_norm, axis=1, keepdims=True)
    # Divide x by its norm.
    x_normalized = x / (x_norm + offset)
    return x_normalized


def encode_image(img_tensor: ImageTensor, model) -> ImageEmbedding:
    print(".", end="")
    with torch.no_grad():
        y = model(img_tensor.tensor)
        hs = y.last_hidden_state.squeeze(0).detach().cpu().numpy()  # (1, 4101, 384) for last hidden state
        # (4101, 384)  = ([batch_size], num_patches + 1 + num_registers, hidden_dim) for DINOv3
    rows, cols = img_tensor.height // img_tensor.patchsize, img_tensor.width // img_tensor.patchsize
    n_patches = rows * cols  # 4096
    # print(f"{hs.shape} - r: {rows}, c: {cols}, h: {img_tensor.height}, w: {img_tensor.width}")
    patch_embs = hs[-n_patches: :].reshape(rows, cols, -1) # (64, 64, 384)
    X = patch_embs.reshape(-1, patch_embs.shape[-1]) # (4096, 384)
    Xn = normalize_embedding(X)  # (4096, 384)
    return ImageEmbedding(img_tensor.image_id, X, Xn, rows, cols, img_tensor.patchsize, img_tensor.source)


def encode(img_data: PreprocessedData, pretrained_model_name: str, device) -> EmbeddingData:
    model = load_encoder_model(pretrained_model_name, device)
    print(f"Encoding {len(raw_img_paths)} preprocessed images ", end="")
    img_embs = [ encode_image(img_tensor, model) for img_tensor in img_data.img_tensors ]
    print(" Done\n")
    return EmbeddingData(img_embs, img_data.n_images, img_data.raw_imgs_metadata, pretrained_model_name)




## Embeddings

In [ ]:

@dataclass
class PatchedImage:
    image_id: str
    image: np.ndarray
    width: int
    height: int
    cols: int
    rows: int
    patchsize: int
    
    def __init__(self, raw_img: RawImage, patchsize: int):
        self.image_id = raw_img.image_id
        self.image = np.array(crop_to_patchsize(load_image(raw_img), patchsize), dtype=np.uint8)  # (H,W,3) for display
        self.height, self.width, self.channels = self.image.shape  # height, width, channels
        self.patchsize = patchsize
        self.rows = self.height // self.patchsize
        self.cols = self.width // self.patchsize
        

    def clamp_idx(self, i: int) -> int: 
        # makes sure i is one of the image patches (in range[0, r*c-1])
        return int(np.clip(i, 0, self.rows * self.cols - 1))
    
    
    def get_n_patches(self) -> int:
        return self.cols * self.rows

### Config

In [ ]:
patchsize = 16
top_n_sim_img_patches = 10
rand_seed = 31


# ---- Images ----
# dir_path = "../data/img/202604-CZSK-IMGs-Stere-Klasse/2026-531-Soobrazitelny-Stere/crops"

# dir_path = "../data/img/2026_CASE_13_STERE_BOIKY"
dir_path = "../data/img/2025_CASE OP_OS_4_STERE_BOIKY"  #1# 66 gedrag - hidden 
# dir_path = "../data/img/2025_CASE_022_STERE_BOIKY"  # 43 heli
# dir_path = "../data/img/2025_CASE_002_STERE_BOIKY"  # 39 achterdek personen, kleur kleding
# dir_path = "../data/img/BOIKY"  # 35  diff - boeg, windvaan, net achterdek
# dir_path = "../data/img/STEREGUSHCHY Korvetten"  ### 34 verschillen brug
n_images = 15
# raw_img_paths = [join(dir_path, f) for f in listdir(dir_path) if isfile(join(dir_path, f)) and f.lower().endswith(("jpg", "png"))][:n_images]
raw_img_paths = list(p.resolve() for p in Path(dir_path).glob("**/*") if p.suffix in {".jpg", ".png", ".tiff"}) # [:n_images]
n_images = len(raw_img_paths)

# ---- Model and device setup ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_model_name = "../models/dinov3-vits16-pretrain-lvd1689m"  # Load locally

# ---- User config ----
show_grid = False
show_overlay = False

overlay_alpha = 0.55
patch_size_override = None  # set to 16 to force; None = read from model if available 

n_disp_imgs = 6
disp_cols = 3
disp_rows = math.ceil(n_disp_imgs/disp_cols)
cmap = plt.get_cmap("magma")

#### Create Embeddings and init Cosine Siliarty data

In [ ]:
def preprocess_and_encode(raw_img_paths: list[str], device, patchsize):
    img_tensors = preprocess(raw_img_paths, device, patchsize)
    img_embs = encode(img_tensors, pretrained_model_name, device)
    return img_embs

embedding_data = preprocess_and_encode(raw_img_paths, device, patchsize)
cos_sim_data = CosineSimilarityData(embedding_data)

### UI

In [ ]:


# ---------- Small drawing utilities ----------

# ---- grid ----
def init_grid(ax, rows, cols, ps) -> list[plt.Line2D]:
    grid = []
    for r in range(1, rows):
        line = ax.axhline(r * ps - 0.5, lw=0.8, alpha=0.6, color="white", zorder=3)
        grid.append(line)
    for c in range(1, cols):
        line = ax.axvline(c * ps - 0.5, lw=0.8, alpha=0.6, color="white", zorder=3)
        grid.append(line)
    return grid

def grid_set_visible(grid: list[plt.Line2D], is_visible: bool):
    for line in grid:
        line.set_visible(is_visible)
    

# def draw_indices(ax, rows, cols, ps):
#     for r in range(rows):
#         for c in range(cols):
#             idx = r * cols + c
#             ax.text(c * ps + ps / 2, r * ps + ps / 2, str(idx),
#                     ha="center", va="center", fontsize=7,
#                     color="white", alpha=0.95, zorder=4)


# ---- red rectangles ----            
def draw_clicked_rect(clicked: ClickedIndices, patchsize: int, red_rects: list[Rectangle]):
    for red_rect in red_rects:
        red_rect.set_visible(False)
    # move rect_object over selected patch of clicked ax
    red_rects[clicked.ax_idx].set_xy((clicked.col_idx * patchsize, clicked.row_idx * patchsize))
    red_rects[clicked.ax_idx].set_visible(True)


# ---- overlay ----
def inflate_to_nearest(x: np.ndarray, h: int, w: int, patchsize: int):
    """Nearest upsample for 2D or 3D arrays with last-dim channels."""
    if x.ndim == 2:
        return x.repeat(patchsize, 0).repeat(patchsize, 1)
    elif x.ndim == 3:
        C = x.shape[-1]
        return x.repeat(patchsize, 0).repeat(patchsize, 1).reshape(h, w, C)
    raise ValueError("Unsupported ndim for upsample")

# ---- titles ----
def set_titles(src_i=None, self_stats=None, cross_stats=None):
    # TODO
    pass


# ---- displayed image indices ----

def get_new_img_idxs(cur_img_idxs: list[int], top_cs_img_idxs: list[int]):
    """Creates the new image_idxs to be displayed using the best image idxs
    while preserving as many images in their current location (list position)
    and creates a change map of the new images and their new position as tuple (position, new_image_index)"""
    
    # find new idxs to add
    add_img_idxs = [i for i in top_cs_img_idxs if i not in cur_img_idxs]
    # find redundant idxs to remove
    rem_img_idxs = [i for i in cur_img_idxs if i not in top_cs_img_idxs]

    new_img_idxs = list(cur_img_idxs)  # copy current
    change_map = []
    for old_img, new_img in zip(rem_img_idxs, add_img_idxs):
        i = cur_img_idxs.index(old_img)
        change_map.append((i, new_img))  # add tuple (index, image_index) for new image to add
        new_img_idxs[i] = new_img   # replace indices of old_img with new_img

    return new_img_idxs, change_map

# ---- overlay  ----

def calculate_overlay(cos_sim_list: np.ndarray, disp_img: PatchedImage):
    
    cos_sim_map = cos_sim_list.reshape(disp_img.rows, disp_img.cols)
    prob_map = (cos_sim_map - cos_sim_map.min()) / (np.ptp(cos_sim_map) + 1e-8)  # push cos_map_self to range [0,1]
    rgba_map = cmap(prob_map)  # assign colors to range range [0,1]
    return inflate_to_nearest(rgba_map, disp_img.height, disp_img.width, disp_img.patchsize)
    # return overlay_img




# 1 Schip **532-Boiky** - 1 Iteratie - 2025 - 66 Foto's

In [ ]:

# ---- all images eligible for display ----
# ---- create display images
def get_display_images(raw_img_paths: list[str], patchsize: int) -> list[PatchedImage]:
    raw_imgs_metadata = [ RawImage(path) for path in raw_img_paths]
    disp_imgs = [ PatchedImage(raw_img, patchsize) for raw_img in raw_imgs_metadata ]
    return disp_imgs

disp_imgs = get_display_images(raw_img_paths, patchsize)

# ---- setup UI ----

# choose the images on display
# disp_img_idxs = list(range(0, min(n_disp_imgs, len(disp_imgs))))  # display first n images
# or random choice
random.seed(rand_seed)
disp_img_idxs = random.sample(range(0, len(disp_imgs)), n_disp_imgs)
print(f"Using a base of {len(disp_imgs)} cropped images (1024x1024)")

# # init cosine similarity data
# cos_sim_data = CosineSimilarityData(embedding_data)

# # selected image and patch (default first image, central patch)
# img_selected_idx = cos_sim_data.img_selected_idx
# patch_selected_idx = cos_sim_data.patch_selected_idx

# cos_sim_data.update_cosine_similarities()
def create_dummy_overlay_np(disp_img: PatchedImage, img_emb: ImageEmbedding):
    init_scalar = 0.5 * np.ones((img_emb.rows, img_emb.cols), dtype=np.float32)
    rgba = cmap(init_scalar)
    rgba_up = inflate_to_nearest(rgba, disp_img.height, disp_img.width, img_emb.patchsize)
    return rgba_up

# images themselves
ground_imgs = []
# similarity overlays
overlay_imgs = []
# grid overlays
overlay_grids = []  
# selected (red) patches (one for each axes)
red_rects = []  
# matching (yellow) patches (one for each axes)
yel_rects = []  
# new (gree) image rects (one for each axes)
grn_rects = []  

# ---- initialize gui components ----
fig, axs = plt.subplots(disp_rows, disp_cols, figsize=(24, 7*disp_rows))  # 17 or 24
for i, (ax, img_idx) in enumerate(zip(axs.flat, disp_img_idxs)):
    ax.set_axis_off()

    disp_img = disp_imgs[img_idx]

    # set ground images
    ground_img = ax.imshow(disp_img.image, zorder=0)  # TODO rename ?
    ground_imgs.append(ground_img)
    
    # TODO replace with display_img ?
    img_emb = embedding_data.img_embs[img_idx]

    ax.set_title(f"{img_emb.source.name}", fontsize=10)

    # create dummy overlay
    overlay_img = ax.imshow(create_dummy_overlay_np(disp_img, img_emb), alpha=0.5, zorder=1)
    overlay_img.set_visible(show_overlay)
    overlay_imgs.append(overlay_img)
    # create selected (red) rectangle
    red_rect = Rectangle((0, 0), patchsize, patchsize, fill=False, lw=2.0, ec="red", visible=False, zorder=5)
    red_rects.append(red_rect)
    ax.add_patch(red_rect)
    # create similar (yellow) rectangles
    yel_img_rects = []
    for i in range(top_n_sim_img_patches):
        yel_rect = Rectangle((0, 0), patchsize, patchsize, fill=False, lw=2.0, ec="yellow", visible=False, zorder=6)
        yel_img_rects.append(yel_rect)
        ax.add_patch(yel_rect)
    yel_rects.append(yel_img_rects)

    # grid
    overlay_grid = init_grid(ax, img_emb.rows, img_emb.cols, img_emb.patchsize)
    grid_set_visible(overlay_grid, show_grid)
    overlay_grids.append(overlay_grid)

# fig.suptitle(f"{dir_path}")

# holder for the active (selected) axes, images and patches 
clicked = ClickedIndices(None, None, None, None, None)

def update_overlays():
    if show_overlay:
        for i in range(len(disp_img_idxs)):
            img_idx = disp_img_idxs[i]
            cos_sim_np = cos_sim_data.patch_img[img_idx].cos_sim
            disp_img = disp_imgs[img_idx]
            overlay_img = calculate_overlay(cos_sim_np, disp_img)

            overlay_imgs[i].set_data(overlay_img)
            overlay_imgs[i].set_alpha(overlay_alpha)

# ---- UI events ----
def on_click(event):
    
    global active
    global axs
    global ground_imgs
    global red_rects
    global disp_img_idxs

    if event.inaxes is None or event.xdata is None or event.ydata is None:
        return  # event happened outside images bounds
    
    # find ax where event happened (if any)
    for i, ax in enumerate(axs.flat):
        if event.inaxes is ax:
            clicked.ax_idx = i
            clicked.img_idx = disp_img_idxs[i]
            break

    if clicked.img_idx is None:
        return
    
    img_emb = embedding_data.img_embs[clicked.img_idx]  # TODO use DisplayImage (renamed to ???)

    # update row, col and patch indices
    clicked.row_idx = img_emb.row(event.ydata)  # use DisplayImage for this
    clicked.col_idx = img_emb.col(event.xdata)
    clicked.update_patch_idx(img_emb.cols)


    # update cos_sim_data for new patch
    cos_sim_data.update(clicked.img_idx, clicked.patch_idx)
    # get top-n image idxs with highest cosine similarity 
    top_n_cs_img_idxs = cos_sim_data.top_n_img_idxs(n_disp_imgs)
    # get new disp_img_idxs and a change map to replace obsolete positions with new image idx
    disp_img_idxs, change_map = get_new_img_idxs(disp_img_idxs, top_n_cs_img_idxs)

    # replace display images and overlays
    for i, img_idx in change_map:
        disp_img = disp_imgs[img_idx]
        ground_imgs[i].set_data(disp_img.image)

    for i, img_idx in enumerate(disp_img_idxs):
        title = disp_imgs[img_idx].image_id.split("/")[-2]
        sim_val = 100*cos_sim_data.patch_img[img_idx].cos_sim.max().item()
        axs.flat[i].set_title(f"{title} • {sim_val:.0f} % similar", fontsize=10)

    update_overlays()
    draw_clicked_rect(clicked, img_emb.patchsize, red_rects)

#     compute_and_show_both_from_src(active_side)

def on_key(event):
    global clicked
    global show_grid
    global show_overlay
    global overlay_alpha

    
    if event.key in ("g", "G"):
        show_grid = not show_grid
        for disp_grid in overlay_grids:
            grid_set_visible(disp_grid, show_grid)
        return
    elif event.key in ("o", "O"):
        show_overlay = not show_overlay
        for overlay_img in overlay_imgs:
            overlay_img.set_visible(show_overlay)
    elif event.key in ("-", "_"):
        overlay_alpha -= 0.05
        overlay_alpha = max(overlay_alpha, 0.0)
    elif event.key in ("+", "="):
        overlay_alpha += 0.05
        overlay_alpha = min(overlay_alpha, 1.0)
    else:
        img_emb = embedding_data.img_embs[clicked.img_idx]
        if event.key == "left":
            clicked.col_idx = max(0, clicked.col_idx-1)
        elif event.key == "right":
            clicked.col_idx = min(img_emb.cols - 1, clicked.col_idx + 1)
        elif event.key == "up":
            clicked.row_idx = max(0, clicked.row_idx - 1)
        elif event.key == "down":
            clicked.row_idx = min(img_emb.rows - 1, clicked.row_idx + 1)
        else:
            return

        clicked.update_patch_idx(img_emb.cols)
        cos_sim_data.update(clicked.img_idx, clicked.patch_idx)

        draw_clicked_rect(clicked, img_emb.patchsize, red_rects)
    update_overlays()
    # current_idx[side] = rc_to_idx(r, c, st["cols"])
    # update_selection_rects()

    # compute_and_show_both_from_src(active_side)

fig.canvas.mpl_connect("button_press_event", on_click)
fig.canvas.mpl_connect("key_press_event", on_key)

fig.tight_layout()
plt.show()